# 16. Production Model — LightGBM + Sequence Features

**Цель:** собрать продовую модель от сырых данных до сохранённого артефакта `.joblib`, готового к деплою в `src/api/`.

**Лучший кандидат (по результатам экспериментов):**

| Параметр | Значение | Откуда |
|---|---|---|
| Модель | LightGBM | NB 13 — лучший AUC + avg/trade среди всех моделей |
| Фичи | 22 base + 80 rolling (5/15/30/60) = **102** | NB 11, NB 13 §9 — rolling окна |
| Торговая логика | `signal_flip` (hold при неопределённости) | NB 13, NB 15 — сравнение с exit_on_05 |
| Порог | **0.75 / 0.25** (band 25-75) | NB 13, NB 15 — лучший avg_%_per_trade |
| Комиссия | 0.1% round-trip | Фиксировано для Bybit |

**Спецификация API:** `ml_api_integration_spec_04-03-2026.md` v2.2.0 (изменения — только AUTH-02, контракт данных не изменился).

**Структура ноутбука:** Setup → Features → Scale → Train → Evaluate → Backtest → Save → Validate

## 1. Импорты и константы

Все параметры модели зафиксированы по итогам экспериментов. Ничего не подбирается — только воспроизводим лучший конфиг.

In [1]:
import sys, os, numpy as np, pandas as pd, joblib, warnings
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report
import lightgbm as lgb

warnings.filterwarnings('ignore')

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == '06_production' else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)

COMMISSION_RT   = 0.001        # 0.1% round-trip (Bybit taker+taker)
THRESHOLD       = 0.75         # band 25-75 → BUY >= 0.75, SELL <= 0.25
SEQ_WINDOWS     = [5, 15, 30, 60]
TARGET_COL      = 'target'
MODEL_PATH      = os.path.join(_root, 'models', 'prod_lgbm_seq.joblib')

print(f'Root: {_root}')
print(f'Threshold: {THRESHOLD} / {1 - THRESHOLD:.2f}')
print(f'Sequence windows: {SEQ_WINDOWS}')
print(f'Model save path: {MODEL_PATH}')

Root: c:\project\trading_bot_2Engine
Threshold: 0.75 / 0.25
Sequence windows: [5, 15, 30, 60]
Model save path: c:\project\trading_bot_2Engine\models\prod_lgbm_seq.joblib


## 2. Загрузка данных и temporal split

Данные подготовлены пайплайном NB 01-03 и сохранены в parquet. В проде обучающие данные загружаются через `GET /api/ml/ds/training-dataset` (spec v2.2.0).

Temporal split 8 / 1 / 1 дней — идентично всем экспериментам (NB 07, 13, 14, 15).

In [2]:
labeled_path = os.path.join(_root, 'outputs', 'data_labeled_tp_sl_1_05.parquet')
feat_path    = os.path.join(_root, 'outputs', 'features_selected_tp_sl_1_05.txt')

df_raw = pd.read_parquet(labeled_path)
with open(feat_path, encoding='utf-8') as f:
    BASE_FEATURES = [l.strip() for l in f if l.strip()]
BASE_FEATURES = [c for c in BASE_FEATURES if c in df_raw.columns]

# Базовая подготовка: target, дата, ret_next внутри session_key
valid = df_raw.dropna(subset=BASE_FEATURES + [TARGET_COL]).copy()
valid = valid[valid[TARGET_COL].isin([-1.0, 1.0])]
valid['y'] = (valid[TARGET_COL] == 1).astype(int)
valid['date'] = pd.to_datetime(valid['datetime'], utc=True).dt.date

sort_col = 'datetime' if 'datetime' in valid.columns else 'timestamp'
valid['ret_next'] = valid.groupby('session_key')['close_price'].pct_change().shift(-1)
valid = valid.dropna(subset=['ret_next'])

# Даты для temporal split (8 / 1 / 1)
dates = sorted(valid['date'].unique())
assert len(dates) >= 10, f'Нужно >= 10 дней, найдено {len(dates)}'

train_dates = set(dates[:8])
val_dates   = set([dates[8]])
test_dates  = set([dates[9]])

# Rolling-фичи считаем на всём valid (как в NB15), затем делаем split
KEY_FEATS = BASE_FEATURES[:10]  # rd_mom_1 .. rsi_14

grp = valid.groupby('session_key', group_keys=False)
for w in SEQ_WINDOWS:
    for c in KEY_FEATS:
        valid[f'{c}_roll{w}_mean'] = grp[c].transform(lambda x: x.rolling(w, min_periods=1).mean())
        valid[f'{c}_roll{w}_std']  = grp[c].transform(lambda x: x.rolling(w, min_periods=1).std().fillna(0))

SEQ_FEATURES = [f'{c}_roll{w}_{s}' for w in SEQ_WINDOWS for c in KEY_FEATS for s in ('mean', 'std')]
ALL_FEATURES = BASE_FEATURES + SEQ_FEATURES

train_df = valid[valid['date'].isin(train_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)
val_df   = valid[valid['date'].isin(val_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)
test_df  = valid[valid['date'].isin(test_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)

print(f'Dates: train={min(train_dates)}..{max(train_dates)} | val={dates[8]} | test={dates[9]}')
print(f'Rows (with rolling):  train={len(train_df):,} | val={len(val_df):,} | test={len(test_df):,}')
print(f'Base features: {len(BASE_FEATURES)}, rolling: {len(SEQ_FEATURES)}, total: {len(ALL_FEATURES)}')

Dates: train=2026-02-01..2026-02-08 | val=2026-02-09 | test=2026-02-10
Rows:  train=146,693 | val=24,014 | test=15,356
Base features: 22


## 3. Проверка базовых фичей

22 базовых фичи уже присутствуют в parquet (рассчитаны через `src/features/feature_pipeline.py` в NB 03, отобраны в NB 05).
В онлайн-инференсе используется та же функция `add_features()` — гарантия идентичности.

In [3]:
missing = [c for c in BASE_FEATURES if c not in train_df.columns]
assert not missing, f'Missing base features: {missing}'

print(f'✓ Все {len(BASE_FEATURES)} базовых фичей на месте')
print('Список:', BASE_FEATURES)

✓ Все 22 базовых фичей на месте
Список: ['rd_mom_1', 'rd_mom_5', 'rd_mom_10', 'rd_acceleration', 'rd_zscore_30', 'rd_ema_20', 'abs_rd', 'ret_1', 'ret_5', 'rsi_14', 'macd_signal', 'macd_hist', 'volatility_14', 'volume_rel_20', 'body_ratio', 'close_position', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'rd_regime', 'rd_regime_transition']


## 4. Rolling sequence features (5, 15, 30, 60)

Первые 10 базовых фичей (rd_mom_1 … rsi_14) агрегируются rolling mean/std по каждому окну внутри `session_key`.
Итого: 10 × 4 окна × 2 (mean, std) = 80 новых → **102 фичи**.

Исследовано в NB 11 (30/60), NB 13 §9 (5/15/30/60 vs 30/60 — новые окна дали лучший avg/trade).

In [4]:
# Проверочный вывод по уже посчитанным rolling-фичам (см. ячейку 2)
missing_seq = [c for c in ALL_FEATURES if c not in train_df.columns]
assert not missing_seq, f'Missing: {missing_seq}'

print(f'Key feats for rolling: {KEY_FEATS}')
print(f'Rolling features added: {len(SEQ_FEATURES)}')
print(f'Total features: {len(ALL_FEATURES)} (22 base + {len(SEQ_FEATURES)} rolling)')

Key feats for rolling: ['rd_mom_1', 'rd_mom_5', 'rd_mom_10', 'rd_acceleration', 'rd_zscore_30', 'rd_ema_20', 'abs_rd', 'ret_1', 'ret_5', 'rsi_14']
Rolling features added: 80
Total features: 102 (22 base + 80 rolling)


## 5. Масштабирование фичей

StandardScaler fit на train, transform на val/test. Скейлер сохраняется в bundle для инференса.
Подход из NB 06 (Scaling_And_Normalization).

In [5]:
scaler = StandardScaler()

X_train = scaler.fit_transform(train_df[ALL_FEATURES].fillna(0))
X_val   = scaler.transform(val_df[ALL_FEATURES].fillna(0))
X_test  = scaler.transform(test_df[ALL_FEATURES].fillna(0))

y_train = train_df['y'].values
y_val   = val_df['y'].values
y_test  = test_df['y'].values

print(f'X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}')
print(f'y balance train: {y_train.mean():.3f} | val: {y_val.mean():.3f} | test: {y_test.mean():.3f}')

X_train: (146693, 102), X_val: (24014, 102), X_test: (15356, 102)
y balance train: 0.501 | val: 0.518 | test: 0.534


## 6. Обучение LightGBM

LightGBM выбран в NB 13 как лучшая модель по совокупности AUC и avg_%_per_trade.
Гиперпараметры из эксперимента NB 13 §9 (без дополнительного тюнинга — воспроизводим лучший конфиг).

In [6]:
model = lgb.LGBMClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    verbose=-1,
)

model.fit(X_train, y_train)

proba_val  = model.predict_proba(X_val)[:, 1]
proba_test = model.predict_proba(X_test)[:, 1]

auc_val  = roc_auc_score(y_val, proba_val)
auc_test = roc_auc_score(y_test, proba_test)

print(f'AUC val:  {auc_val:.6f}')
print(f'AUC test: {auc_test:.6f}')

AUC val:  0.907077
AUC test: 0.806928


## 7. Оценка качества классификации

Порог 0.75 (25-75) выбран в NB 13, NB 15 как оптимальный по avg_%_per_trade.
Classification report показывает precision/recall при этом пороге.

In [7]:
y_pred_test = (proba_test >= THRESHOLD).astype(int)

print(f'=== Classification Report (test, threshold={THRESHOLD}) ===')
print(classification_report(y_test, y_pred_test, digits=4))
print(f'AUC val:  {auc_val:.6f}')
print(f'AUC test: {auc_test:.6f}')

=== Classification Report (test, threshold=0.75) ===
              precision    recall  f1-score   support

           0     0.5366    0.9734    0.6918      7154
           1     0.9201    0.2666    0.4135      8202

    accuracy                         0.5959     15356
   macro avg     0.7283    0.6200    0.5526     15356
weighted avg     0.7414    0.5959    0.5431     15356

AUC val:  0.907077
AUC test: 0.806928


## 8. Бэктест на тестовом дне

Логика `signal_flip` (prod_hold): BUY >= 0.75, SELL <= 0.25, HOLD — сохраняем предыдущую позицию.
Комиссия 0.1% round-trip. Идентично NB 08, NB 13.

In [8]:
def backtest_pnl(proba, ret, session_ids, threshold=THRESHOLD, commission_rt=COMMISSION_RT):
    """signal_flip: BUY >= thr, SELL <= 1-thr, HOLD keeps previous position."""
    pred = np.where(proba >= threshold, 1, np.where(proba <= 1 - threshold, 0, -1))
    n = len(proba)
    pos = np.zeros(n, dtype=np.float64)
    prev = 0.0
    for i in range(n):
        if session_ids is not None and i > 0 and session_ids[i] != session_ids[i - 1]:
            prev = 0.0
        if pred[i] == 1:
            pos[i], prev = 1.0, 1.0
        elif pred[i] == 0:
            pos[i], prev = -1.0, -1.0
        else:
            pos[i] = prev
    pos_prev = np.roll(pos, 1)
    pos_prev[0] = 0.0
    sess_chg = np.zeros(n, dtype=bool)
    if session_ids is not None:
        sess_chg[1:] = session_ids[1:] != session_ids[:-1]
    pos_prev = np.where(sess_chg, 0.0, pos_prev)
    pos_changed = (pos != pos_prev) & ((pos != 0) | (pos_prev != 0))
    fee_total = pos_changed.sum() * (commission_rt / 2.0)
    pnl_net = (pos * ret).sum() - fee_total
    trades = int(pos_changed.sum())
    avg_trade = float((pnl_net * 100) / trades) if trades > 0 else np.nan
    return {'trades': trades, 'net_%': float(pnl_net * 100), 'avg_%_per_trade': avg_trade}

bt_test = backtest_pnl(proba_test, test_df['ret_next'].values, test_df['session_key'].values)
bt_val  = backtest_pnl(proba_val,  val_df['ret_next'].values,  val_df['session_key'].values)

print('=== Backtest Results (signal_flip, threshold 25-75) ===')
print(f'VAL  day: net={bt_val["net_%"]:+.2f}%  trades={bt_val["trades"]}  avg/trade={bt_val["avg_%_per_trade"]:.4f}%')
print(f'TEST day: net={bt_test["net_%"]:+.2f}%  trades={bt_test["trades"]}  avg/trade={bt_test["avg_%_per_trade"]:.4f}%')

=== Backtest Results (signal_flip, threshold 25-75) ===
VAL  day: net=+2554.15%  trades=1559  avg/trade=1.6383%
TEST day: net=+1146.17%  trades=840  avg/trade=1.3645%


## 9. Сохранение продового артефакта

Bundle содержит обязательные ключи для `src/api/model_bundle.py` (`model`, `scaler`, `features`) + дополнительные для воспроизводимости и будущего инференса с rolling-фичами.

In [9]:
bundle = {
    'model':          model,
    'model_name':     'LightGBM_seq_5_15_30_60',
    'scaler':         scaler,
    'features':       ALL_FEATURES,
    'base_features':  BASE_FEATURES,
    'seq_windows':    SEQ_WINDOWS,
    'seq_key_feats':  KEY_FEATS,
    'target':         TARGET_COL,
    'threshold':      THRESHOLD,
    'threshold_lo':   1 - THRESHOLD,
    'commission_rt':  COMMISSION_RT,
    'val_auc':        auc_val,
    'test_auc':       auc_test,
    'backtest_test':  bt_test,
    'backtest_val':   bt_val,
    'split': {
        'type': 'temporal',
        'train_days': 8,
        'val_days': 1,
        'test_days': 1,
        'train_range': f'{min(train_dates)}..{max(train_dates)}',
        'val_date': str(dates[8]),
        'test_date': str(dates[9]),
    },
    'spec_version': '2.2.0',
}

joblib.dump(bundle, MODEL_PATH, compress=3)

fsize = os.path.getsize(MODEL_PATH) / (1024 * 1024)
print(f'Saved: {MODEL_PATH} ({fsize:.1f} MB)')
print(f'Bundle keys: {list(bundle.keys())}')

Saved: c:\project\trading_bot_2Engine\models\prod_lgbm_seq.joblib (0.5 MB)
Bundle keys: ['model', 'model_name', 'scaler', 'features', 'base_features', 'seq_windows', 'seq_key_feats', 'target', 'threshold', 'threshold_lo', 'commission_rt', 'val_auc', 'test_auc', 'backtest_test', 'backtest_val', 'split', 'spec_version']


## 10. Валидация артефакта

Загружаем через `load_model_bundle()` — тот же путь, что использует API.
Smoke-test: предсказание на одном сэмпле, проверка сигнала.

In [10]:
from src.api.model_bundle import load_model_bundle

loaded = load_model_bundle(MODEL_PATH)

assert loaded['model_name'] == 'LightGBM_seq_5_15_30_60'
assert len(loaded['features']) == len(ALL_FEATURES)
assert loaded['threshold'] == THRESHOLD
print(f'✓ Bundle loaded, model={loaded["model_name"]}, features={len(loaded["features"])}')

sample = X_test[:1]
p = loaded['model'].predict_proba(loaded['scaler'].transform(
    test_df[loaded['features']].fillna(0).iloc[:1]
))[:, 1][0]

thr_hi = loaded['threshold']
thr_lo = loaded.get('threshold_lo', 1 - thr_hi)
if p >= thr_hi:
    sig = 'BUY'
elif p <= thr_lo:
    sig = 'SELL'
else:
    sig = 'HOLD'

print(f'Smoke test: proba={p:.4f}, signal={sig} (thr_hi={thr_hi}, thr_lo={thr_lo})')
print('✓ Артефакт валиден и совместим с src/api/model_bundle.py')

✓ Bundle loaded, model=LightGBM_seq_5_15_30_60, features=102
Smoke test: proba=0.3072, signal=HOLD (thr_hi=0.75, thr_lo=0.25)
✓ Артефакт валиден и совместим с src/api/model_bundle.py


## 11. Итоги

### Продовая модель

| Параметр | Значение |
|---|---|
| Модель | LightGBM (n_est=300, depth=6, lr=0.05) |
| Фичи | 102 (22 base + 80 rolling 5/15/30/60) |
| Порог | 0.75 / 0.25 (band 25-75) |
| Торговая логика | signal_flip (HOLD сохраняет позицию) |
| Комиссия | 0.1% round-trip |
| Артефакт | `models/prod_lgbm_seq.joblib` |
| Спецификация API | v2.2.0 (AUTH-02, данные без изменений) |

### Метрики

Значения AUC val/test и бэктеста — см. вывод ячеек 6-8 выше.

### Что входит в bundle

- `model` — обученный LGBMClassifier
- `scaler` — StandardScaler (fit на train)
- `features` — список 102 фичей (порядок важен)
- `base_features` — 22 исходных фичи (для `add_features()`)
- `seq_windows` — [5, 15, 30, 60]
- `seq_key_feats` — 10 фичей для rolling агрегации
- `threshold` / `threshold_lo` — пороги BUY/SELL
- `split`, `val_auc`, `test_auc`, `backtest_*` — метаданные

### Следующие шаги

1. **Обновить `inference_service.py`** — добавить расчёт rolling-фичей (80 новых) при онлайн-инференсе. Текущий `add_features()` считает только 22 base. Для rolling нужен буфер последних 60 точек per session.
2. **Настроить Basic Auth** — spec v2.2.0 вводит профили `operator` / `dataset`. Для поллинга и ingest использовать `operator`.
3. **Shadow deploy** — запустить с `source=ml_shadow`, мониторить сигналы vs реальная динамика.
4. **Расширить обучающую выборку** — текущая модель обучена на 8 днях. По мере накопления данных через DS API переобучать.